# Build fixed_batches_20

Reads all 20 batches from `sorted_batches_20_reprod_annot` and writes a cleaned version to `fixed_batches_20`.

Each output file keeps only:
```json
{"id": ..., "specie": ..., "specie_en": ..., "fixed_text": ...}
```

Source mapping:
| Source folder | `fixed_text` taken from |
|---|---|
| `without_reprod_annot` | `corrected_text` |
| `without_errors` | `corrected_text` |
| `with_errors` | `corrected_text_final` |

In [13]:
import json
from pathlib import Path

src_base = Path("../sorted_batches_20_reprod_annot")
dst_base = Path("../fixed_batches_20_sorted_GSI")
#dst_base = Path("../fixed_batches_20")

# source_folder -> key to use as fixed_text
folder_key_map = {
    "without_reprod_annot": "corrected_text",
    "without_errors":       "corrected_text",
    "with_errors":          "corrected_text_final",
}

stats = []  # for the summary table

for batch_dir in sorted(src_base.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue

    batch_stats = {"batch": batch_dir.name}

    for folder, text_key in folder_key_map.items():
        src_folder = batch_dir / folder
        dst_folder = dst_base / batch_dir.name / folder
        dst_folder.mkdir(parents=True, exist_ok=True)

        count = 0
        if src_folder.exists():
            for src_file in sorted(src_folder.glob("*.json")):
                with open(src_file, "r", encoding="utf-8") as f:
                    data = json.load(f)

                out = {
                    "id":         data["id"],
                    "specie":     data["specie"],
                    "specie_en":  data["specie_en"],
                    "fixed_text": data[text_key],
                }

                dst_file = dst_folder / src_file.name
                with open(dst_file, "w", encoding="utf-8") as f:
                    json.dump(out, f, ensure_ascii=False, indent=2)

                count += 1

        batch_stats[folder] = count

    batch_stats["total"] = sum(batch_stats[f] for f in folder_key_map)
    stats.append(batch_stats)
    print(f"{batch_dir.name}: {batch_stats['total']} files written")

print(f"\nDone. Output: {dst_base.resolve()}")

batch_01: 370 files written
batch_02: 370 files written
batch_03: 370 files written
batch_04: 370 files written
batch_05: 370 files written
batch_06: 370 files written
batch_07: 370 files written
batch_08: 370 files written
batch_09: 370 files written
batch_10: 370 files written
batch_11: 370 files written
batch_12: 370 files written
batch_13: 370 files written
batch_14: 370 files written
batch_15: 370 files written
batch_16: 370 files written
batch_17: 370 files written
batch_18: 370 files written
batch_19: 370 files written
batch_20: 251 files written

Done. Output: /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/fixed_batches_20_sorted_GSI


In [14]:
import pandas as pd

df = pd.DataFrame(stats)
df.columns = ["Batch", "without_reprod_annot", "without_errors", "with_errors", "Total"]

totals = df[["without_reprod_annot", "without_errors", "with_errors", "Total"]].sum().to_dict()
totals["Batch"] = "TOTAL"
df_display = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

df_display

,Batch,without_reprod_annot,without_errors,with_errors,Total
0,batch_01,52,59,259,370
1,batch_02,57,67,246,370
2,batch_03,47,63,260,370
3,batch_04,70,48,252,370
4,batch_05,45,58,267,370
5,batch_06,67,72,231,370
6,batch_07,56,75,239,370
7,batch_08,41,63,266,370
8,batch_09,63,57,250,370
9,batch_10,64,63,243,370


In [15]:
import shutil

# Merge all 3 subfolders into the flat batch folder, then remove the subfolders
for batch_dir in sorted(dst_base.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue

    for subfolder in list(batch_dir.iterdir()):
        if not subfolder.is_dir():
            continue
        for f in subfolder.glob("*.json"):
            f.rename(batch_dir / f.name)  # move file up one level
        subfolder.rmdir()  # remove now-empty subfolder

    flat_count = len(list(batch_dir.glob("*.json")))
    print(f"{batch_dir.name}: {flat_count} files")

print(f"\nDone. All files are now flat inside each batch folder.")

batch_01: 370 files
batch_02: 370 files
batch_03: 370 files
batch_04: 370 files
batch_05: 370 files
batch_06: 370 files
batch_07: 370 files
batch_08: 370 files
batch_09: 370 files
batch_10: 370 files
batch_11: 370 files
batch_12: 370 files
batch_13: 370 files
batch_14: 370 files
batch_15: 370 files
batch_16: 370 files
batch_17: 370 files
batch_18: 370 files
batch_19: 370 files
batch_20: 251 files

Done. All files are now flat inside each batch folder.


In [11]:
# Reorganise: move each file into a species subfolder named after the species
# fixed_batches_20/batch_01/Ablennes_hians/0003_Ablennes_hians_abj.json

for batch_dir in sorted(dst_base.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue

    for json_file in list(batch_dir.glob("*.json")):
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        species_name = data["specie"]          # e.g. "Ablennes_hians"
        species_dir = batch_dir / species_name
        species_dir.mkdir(exist_ok=True)

        json_file.rename(species_dir / json_file.name)

    n_species = sum(1 for d in batch_dir.iterdir() if d.is_dir())
    print(f"{batch_dir.name}: {n_species} species folders created")

print(f"\nDone. Structure: fixed_batches_20/batch_XX/<species>/<file>.json")

batch_01: 370 species folders created
batch_02: 370 species folders created
batch_03: 370 species folders created
batch_04: 370 species folders created
batch_05: 370 species folders created
batch_06: 370 species folders created
batch_07: 370 species folders created
batch_08: 370 species folders created
batch_09: 370 species folders created
batch_10: 370 species folders created
batch_11: 370 species folders created
batch_12: 370 species folders created
batch_13: 370 species folders created
batch_14: 370 species folders created
batch_15: 370 species folders created
batch_16: 370 species folders created
batch_17: 370 species folders created
batch_18: 370 species folders created
batch_19: 370 species folders created
batch_20: 251 species folders created

Done. Structure: fixed_batches_20/batch_XX/<species>/<file>.json


In [12]:
import shutil

species_data_dir = Path("../../data-generation/species_data")

copied, missing = 0, []

for batch_dir in sorted(dst_base.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith("batch_"):
        continue

    for species_dir in sorted(batch_dir.iterdir()):
        if not species_dir.is_dir():
            continue

        species_name = species_dir.name          # e.g. "Ablennes_hians"
        src_file = species_data_dir / f"{species_name}.json"

        if src_file.exists():
            shutil.copy2(src_file, species_dir / src_file.name)
            copied += 1
        else:
            missing.append(f"{batch_dir.name}/{species_name}")

print(f"Copied : {copied} species_data files")
if missing:
    print(f"Missing: {len(missing)} files not found in species_data:")
    for m in missing:
        print(f"  {m}")
else:
    print("All species data files found and copied.")

Copied : 7281 species_data files
All species data files found and copied.
